# Quick Reinit - Run this after kernel restart to restore key objects



In [1]:
import sys, os, json, yaml, logging, time
from pathlib import Path

# Setup paths & config
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root / 'src'))

# Load config
config_path = project_root / 'config' / 'settings.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Reload key components
from ingestion import DataCleaner
from embedding import EmbeddingManager
from retrieval import RAGRetriever, ResultRanker
from rag import RAGGraph, AnswerGenerator

print("Reinitializing RAG pipeline...")

# Load cleaned products
cleaned_products = json.load(open(project_root / config['data']['processed_dir'] / 'products_clean.json'))

# Reload embedder & retriever
embedder = EmbeddingManager(
    model_name=config['embedding']['model_name'],
    db_path=str(project_root / config['data']['vector_db_dir']),
    collection_name=config['vector_db']['collection_name'],
    device=config['embedding']['device']
)

retriever = RAGRetriever(
    embedding_model=config['embedding']['model_name'],
    db_path=str(project_root / config['data']['vector_db_dir']),
    collection_name=config['vector_db']['collection_name'],
    dense_weight=config['retrieval']['dense_weight'],
    bm25_weight=config['retrieval']['bm25_weight']
)

ranker = ResultRanker(
    use_cross_encoder=config['retrieval']['use_cross_encoder'],
    model_name=config['retrieval'].get('cross_encoder_model')
)

generator = AnswerGenerator(
    llm_provider=config['llm']['provider'],
    model=config['llm']['model'],
    temperature=config['llm']['temperature'],
    max_tokens=config['llm']['max_tokens']
)

rag_graph = RAGGraph(
    retriever=retriever,
    ranker=ranker,
    generator=generator,
    config_path=str(project_root / 'config' / 'prompts.yaml')
)

print("✅ RAG pipeline restored")

/Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reinitializing RAG pipeline...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5792.69it/s]
/Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/src/retrieval/retriever.py:38: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9054.23it/s]


✅ RAG pipeline restored


/Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/src/retrieval/retriever.py:41: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_store = Chroma(


# IKEA RAG Pipeline - Phase-by-Phase Execution

This notebook executes each phase of the RAG pipeline individually, allowing you to:
- Execute phases one at a time
- Inspect intermediate results
- Debug and iterate
- Visualize data at each stage

**Phases:**
1. Setup & Configuration
2. Data Ingestion & Cleaning
3. Chunking
4. Embedding & Vector DB
5. Retrieval Setup
6. RAG Pipeline
7. Query Testing
8. Evaluation (Optional)


## Setup: Imports & Configuration

In [2]:
import sys
import os
from pathlib import Path
import json
import logging
from dotenv import load_dotenv
import yaml

# Setup path
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root / 'src'))

# Load environment variables
load_dotenv(project_root / '.env')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print(f"Project root: {project_root}")
print(f"ANTHROPIC_API_KEY set: {'ANTHROPIC_API_KEY' in os.environ}")

Project root: /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog
ANTHROPIC_API_KEY set: True


## Load Configuration

In [3]:
# Load configuration
config_path = project_root / 'config' / 'settings.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully")
print(f"\nKey Settings:")
print(f"  Dataset: {config['dataset']['repo_id']}")
print(f"  Embedding Model: {config['embedding']['model_name']}")
print(f"  Chunk Size: {config['chunking']['chunk_size']}")
print(f"  LLM: {config['llm']['provider']} - {config['llm']['model']}")
print(f"  Vector DB: {config['vector_db']['collection_name']}")

Configuration loaded successfully

Key Settings:
  Dataset: crawlfeeds/IKEA-Home-Decor-Furniture-Dataset
  Embedding Model: sentence-transformers/all-MiniLM-L6-v2
  Chunk Size: 512
  LLM: anthropic - claude-sonnet-5
  Vector DB: ikea_products


---
# Phase 1: Data Ingestion

Download IKEA dataset from Hugging Face Hub and save locally.

In [7]:
from ingestion import IKEADatasetDownloader, DataCleaner

print("Phase 1: Data Ingestion")
print("="*50)

# Initialize downloader
downloader = IKEADatasetDownloader(
    repo_id=config['dataset']['repo_id'],
    cache_dir=str(project_root / config['data']['raw_dir']),
    images_dir=str(project_root / config['data']['images_dir'])
)

print("Downloading dataset from Hugging Face...")
print("This may take 2-5 minutes depending on internet speed.")

# Download dataset
dataset = downloader.download_dataset(split=config['dataset']['split'])

print(f"\n✅ Dataset downloaded successfully")
print(f"   Total products: {len(dataset)}")
print(f"   Sample product keys: {list(dataset[0].keys())[:5]}")


2026-08-28 18:52:44,744 - ingestion.downloader - INFO - Loading dataset crawlfeeds/IKEA-Home-Decor-Furniture-Dataset from Hugging Face Hub...


Phase 1: Data Ingestion
This may take 2-5 minutes depending on internet speed.


2026-08-28 18:52:48,596 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/crawlfeeds/IKEA-Home-Decor-Furniture-Dataset/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-08-28 18:52:48,613 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/crawlfeeds/IKEA-Home-Decor-Furniture-Dataset/80698456440f91df73c01a83bd60b3a49074c2f4/README.md "HTTP/1.1 200 OK"
2026-08-28 18:52:48,707 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/crawlfeeds/IKEA-Home-Decor-Furniture-Dataset/resolve/80698456440f91df73c01a83bd60b3a49074c2f4/IKEA-Home-Decor-Furniture-Dataset.py "HTTP/1.1 404 Not Found"
2026-08-28 18:53:17,438 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/crawlfeeds/IKEA-Home-Decor-Furniture-Dataset/crawlfeeds/IKEA-Home-Decor-Furniture-Dataset.py "HTTP/1.1 404 Not Found"
2026-08-28 18:53:17,937 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datase


✅ Dataset downloaded successfully
   Total products: 464
   Sample product keys: ['product_url', 'country', 'product_name', 'product_id', 'product_type']


## Save Raw Products

In [6]:
# Save raw products as JSON
raw_output = project_root / config['data']['raw_dir'] / 'ikea_products.json'

print(f"Saving raw products to {raw_output}...")
downloader.process_and_save(dataset, str(raw_output))

# Check saved file
with open(raw_output, 'r') as f:
    products = json.load(f)

print(f"\n✅ Saved {len(products)} products")
print(f"\nSample product:")
print(json.dumps(products[0], indent=2))

2026-08-28 18:52:20,876 - ingestion.downloader - INFO - Processing dataset...


Saving raw products to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/raw/ikea_products.json...


Processing products: 100%|██████████| 464/464 [00:00<00:00, 5956.12it/s]
2026-08-28 18:52:20,963 - ingestion.downloader - INFO - Saved 464 products to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/raw/ikea_products.json
2026-08-28 18:52:20,964 - ingestion.downloader - INFO - Saved manifest to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/raw/data_manifest.json



✅ Saved 464 products

Sample product:
{
  "product_id": "60502107",
  "name": "ADLAD",
  "category": "Home Decor",
  "price": 186,
  "description": "Scented candle in glass, Scandinavian Woods/white",
  "materials": "Material:\nGlass\nPlant based wax (min. 50%), paraffin wax, fragrance\nCotton and other cellulose based materials\n\nCare:\nHandwash only.\nYou easily remove hardened wax by washing the surface with warm water.",
  "dimensions": "Burning time: 20 hr | Diameter: 7.5 cm (3 \") | Height: 7 cm (2 \u00be \")",
  "primary_image": "https://www.ikea.com/in/en/images/products/adlad-scented-candle-in-glass-scandinavian-woods-white__1245091_pe921480_s5.jpg",
  "additional_images": "[\"https://www.ikea.com/in/en/images/products/adlad-scented-candle-in-glass-scandinavian-woods-white__1245091_pe921480_s5.jpg\", \"https://www.ikea.com/pvid/1227143_fe002614.mp4\", \"https://www.ikea.com/in/en/images/products/adlad-scented-candle-in-glass-scandinavian-woods-white__1079922_pe857778_s5.jpg\

# Phase 1b: Download Images (self-contained)

In [8]:
# Phase 1b: Download Images (self-contained)
from ingestion import IKEADatasetDownloader
import json

print("Phase 1b: Downloading Product Images")
print("="*50)

# Define paths
raw_output = project_root / config['data']['raw_dir'] / 'ikea_products.json'

# Load products
with open(raw_output, 'r') as f:
    products = json.load(f)

print(f"Loading {len(products)} products...")

# Download images
downloader = IKEADatasetDownloader(
    repo_id=config['dataset']['repo_id'],
    cache_dir=str(project_root / config['data']['raw_dir']),
    images_dir=str(project_root / config['data']['images_dir'])
)

result = downloader.download_images(products)
products = result['products']

# Save updated products with image paths
with open(raw_output, 'w') as f:
    json.dump(products, f, indent=2)

print(f"\n✅ Image Download Complete:")
print(f"   Downloaded: {result['stats']['downloaded']}")
print(f"   Failed: {result['stats']['failed']}")
print(f"   Skipped: {result['stats']['skipped']}")


2026-08-28 18:55:16,204 - ingestion.downloader - INFO - Starting image download for 464 products...
2026-08-28 18:55:16,205 - ingestion.image_downloader - INFO - Downloading images for 464 products...


Phase 1b: Downloading Product Images
Loading 464 products...


2026-08-28 18:57:26,471 - ingestion.image_downloader - INFO - Download complete: 464 downloaded, 0 failed, 0 skipped
2026-08-28 18:57:26,473 - ingestion.image_downloader - INFO - Saved download log to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/raw/images/download_log.json
2026-08-28 18:57:26,473 - ingestion.downloader - INFO - Image download complete: 464 downloaded, 0 failed



✅ Image Download Complete:
   Downloaded: 464
   Failed: 0
   Skipped: 0


---
# Phase 2: Data Cleaning

Clean text, normalize prices, handle missing values, generate quality reports.

In [3]:
# Phase 2: Data Cleaning (self-contained)
from ingestion import DataCleaner
import json

print("Phase 2: Data Cleaning")
print("="*50)

# Define paths
raw_output = project_root / config['data']['raw_dir'] / 'ikea_products.json'
cleaned_output = project_root / config['data']['processed_dir'] / 'products_clean.json'

# Initialize cleaner
cleaner = DataCleaner()

# Load raw products
raw_products = cleaner.load_products(str(raw_output))

# Clean products
print("Cleaning products...")
cleaned_products = cleaner.clean_products(raw_products)

print(f"\n✅ Cleaning complete")
print(f"   Input: {len(raw_products)} products")
print(f"   Output: {len(cleaned_products)} products")
print(f"   Removed: {len(raw_products) - len(cleaned_products)} products")
print(f"   Cleaning stats: {cleaner.cleaning_stats}")

# Save cleaned products
cleaner.save_cleaned_products(cleaned_products, str(cleaned_output))

print(f"✅ Saved cleaned products to {cleaned_output}")

# Verify image_path is preserved
sample = cleaned_products[0]
print(f"\n✅ Sample product has image_path? {'image_path' in sample}")
print(f"   Value: {sample.get('image_path', 'MISSING')}")


Phase 2: Data Cleaning
Cleaning products...


Cleaning: 100%|██████████| 464/464 [00:00<00:00, 53583.62it/s]


✅ Cleaning complete
   Input: 464 products
   Output: 464 products
   Removed: 0 products
   Cleaning stats: {'total_input': 464, 'duplicates_removed': 0, 'empty_names_removed': 0, 'total_output': 464}
✅ Saved cleaned products to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/processed/products_clean.json

✅ Sample product has image_path? True
   Value: /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/raw/images/60502107.jpg


In [10]:
# Check raw product structure
raw_products = cleaner.load_products(str(raw_output))
print("Sample raw product:")
print(json.dumps(raw_products[0], indent=2))
print("\nAll available keys:")
print(raw_products[0].keys())

2026-08-28 10:47:16,331 - ingestion.cleaner - INFO - Loading products from /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/raw/ikea_products.json
2026-08-28 10:47:16,338 - ingestion.cleaner - INFO - Loaded 464 products


Sample raw product:
{
  "product_id": "60502107",
  "name": "",
  "category": "",
  "price": 186,
  "description": "Scented candle in glass, Scandinavian Woods/white",
  "materials": "",
  "dimensions": "",
  "image_url": "",
  "availability": "",
  "rating": null
}

All available keys:
dict_keys(['product_id', 'name', 'category', 'price', 'description', 'materials', 'dimensions', 'image_url', 'availability', 'rating'])


## Save Cleaned Products & Quality Report

In [4]:
# Save cleaned products
cleaned_output = project_root / config['data']['processed_dir'] / 'products_clean.json'

cleaner.save_cleaned_products(cleaned_products, str(cleaned_output))

# Generate quality report
report = cleaner.generate_quality_report(cleaned_products)

report_output = project_root / config['data']['processed_dir'] / 'quality_report.json'
with open(report_output, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✅ Saved cleaned products to {cleaned_output}")
print(f"\nData Quality Report:")
for key, value in report.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")

# Show sample cleaned product
print(f"\nSample cleaned product:")
print(json.dumps(cleaned_products[0], indent=2))

✅ Saved cleaned products to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/processed/products_clean.json

Data Quality Report:
  total_products: 464
  columns: ['product_id', 'name', 'category', 'price', 'description', 'materials', 'dimensions', 'primary_image', 'image_path', 'product_url', 'availability', 'rating']
  missing_values:
    product_id: 0
    name: 0
    category: 0
    price: 0
    description: 0
    materials: 0
    dimensions: 0
    primary_image: 0
    image_path: 0
    product_url: 0
    availability: 0
    rating: 90
  price_stats:
    min: 39.0
    max: 14970.0
    mean: 1640.1961206896551
  availability_distribution:
    unknown: 464
  categories: 1
  avg_description_length: 23

Sample cleaned product:
{
  "product_id": "60502107",
  "name": "ADLAD",
  "category": "Home Decor",
  "price": 186.0,
  "description": "Scented candle in glass, Scandinavian Woods/white",
  "materials": "Material: Glass Plant based wax

---
# Phase 3: Chunking

Split products into semantic chunks with metadata preservation.

In [5]:
from chunking import TextChunker

print("Phase 3: Chunking")
print("="*50)

# Initialize chunker
chunker = TextChunker(
    chunk_size=config['chunking']['chunk_size'],
    chunk_overlap=config['chunking']['chunk_overlap']
)

# Create chunks
print(f"Chunking {len(cleaned_products)} products...")
print(f"Chunk size: {config['chunking']['chunk_size']} tokens")
print(f"Overlap: {config['chunking']['chunk_overlap']} tokens")

chunks = chunker.chunk_products(cleaned_products)

print(f"\n✅ Chunking complete")
print(f"   Total chunks: {len(chunks)}")
print(f"   Avg chunks per product: {len(chunks) / len(cleaned_products):.1f}")

Phase 3: Chunking
Chunking 464 products...
Chunk size: 512 tokens
Overlap: 100 tokens


Creating chunks: 100%|██████████| 464/464 [00:00<00:00, 102171.20it/s]


✅ Chunking complete
   Total chunks: 533
   Avg chunks per product: 1.1


## Analyze Chunk Statistics

In [6]:
# Get statistics
stats = chunker.get_chunk_statistics(chunks)

print("Chunk Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.1f}")
    else:
        print(f"  {key}: {value}")

# Show sample chunks
print(f"\nSample chunks for first product:")
product_chunks = [c for c in chunks if c['product_id'] == chunks[0]['product_id']]
for i, chunk in enumerate(product_chunks[:2]):
    print(f"\nChunk {i}:")
    print(f"  Product: {chunk['product_name']}")
    print(f"  Category: {chunk['category']}")
    print(f"  Price: ${chunk['price']}")
    print(f"  Text: {chunk['text'][:200]}...")

Chunk Statistics:
  total_chunks: 533
  avg_chunk_length: 54.4
  min_chunk_length: 6
  max_chunk_length: 101
  std_chunk_length: 23.9

Sample chunks for first product:

Chunk 0:
  Product: ADLAD
  Category: Home Decor
  Price: $186.0
  Text: Product: ADLAD
Category: Home Decor
Description: Scented candle in glass, Scandinavian Woods/white
Materials: Material: Glass Plant based wax (min. 50%), paraffin wax, fragrance Cotton and other cellu...


## Save Chunks

In [7]:
# Save chunks
chunks_output = project_root / config['data']['processed_dir'] / 'chunks.json'

chunker.save_chunks(chunks, str(chunks_output))

print(f"✅ Saved {len(chunks)} chunks to {chunks_output}")

✅ Saved 533 chunks to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/data/processed/chunks.json



# Phase 4: Embedding & Vector Database

Generate embeddings and populate ChromaDB vector database.

In [8]:
from embedding import EmbeddingManager

print("Phase 4: Embedding & Vector Database")
print("="*50)

# Initialize embedding manager
print(f"Loading embedding model: {config['embedding']['model_name']}")
print(f"This may take 1-2 minutes on first run...")

embedder = EmbeddingManager(
    model_name=config['embedding']['model_name'],
    db_path=str(project_root / config['data']['vector_db_dir']),
    collection_name=config['vector_db']['collection_name'],
    device=config['embedding']['device']
)

print(f"\n✅ Embedding model loaded")

Phase 4: Embedding & Vector Database
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
This may take 1-2 minutes on first run...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7949.46it/s]



✅ Embedding model loaded


## Add Chunks to Vector Database

In [9]:
# Add chunks to ChromaDB
print(f"Adding {len(chunks)} chunks to ChromaDB...")
print(f"Batch size: {config['embedding']['batch_size']}")
print(f"This may take 5-15 minutes depending on hardware...\n")

embedder.add_chunks_to_db(chunks, batch_size=config['embedding']['batch_size'])

print(f"\n✅ Chunks added to ChromaDB")

Adding 533 chunks to ChromaDB...
Batch size: 32
This may take 5-15 minutes depending on hardware...



Adding to ChromaDB: 100%|██████████| 17/17 [00:00<00:00, 19.52it/s]


✅ Chunks added to ChromaDB


## Verify Vector Database

In [10]:
# Get collection stats
stats = embedder.get_collection_stats()

print("Vector Database Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

# Test retrieval
print(f"\nTesting retrieval with sample query...")
test_results = embedder.query("comfortable couch", top_k=3)

print(f"\nRetrieved {len(test_results['documents'])} results:")
for i, (doc, meta, dist) in enumerate(zip(
    test_results['documents'],
    test_results['metadatas'],
    test_results['distances']
)):
    print(f"\n{i+1}. {meta.get('product_name', 'Unknown')}")
    print(f"   Category: {meta.get('category', 'N/A')}")
    print(f"   Price: ${meta.get('price', 'N/A')}")
    print(f"   Distance: {dist:.4f}")
    print(f"   Text: {doc[:150]}...")

/Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/src/embedding/embedder.py:132: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "embedding_dimension": self.model.get_sentence_embedding_dimension(),


Vector Database Statistics:
  collection_name: ikea_products
  total_chunks: 533
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
  embedding_dimension: 384

Testing retrieval with sample query...

Retrieved 3 results:

1. LINDBYN
   Category: Home Decor
   Price: $2490.0
   Distance: 0.6669
   Text: Product: LINDBYN
Category: Home Decor
Description: Mirror, black
Materials: Material: Glass, Plastic foil Aluminium (min. 70% recycled), Anodized Care...

2. TOFTBYN
   Category: Home Decor
   Price: $7990.0
   Distance: 0.6686
   Text: Product: TOFTBYN
Category: Home Decor
Description: Mirror, white
Materials: Material: Fibreboard, Plastic foil Glass, Plastic foil Care: Wipe clean wi...

3. LINDBYN
   Category: Home Decor
   Price: $1490.0
   Distance: 0.6699
   Text: Product: LINDBYN
Category: Home Decor
Description: Table mirror, black
Materials: Material: Glass, Plastic foil Glass Aluminium (min. 70% recycled), A...


---
# Phase 5: Retrieval Setup

Configure hybrid retrieval (dense + BM25).

In [11]:
from retrieval import RAGRetriever, ResultRanker

print("Phase 5: Retrieval Setup")
print("="*50)

# Initialize retriever
print("Initializing hybrid retriever...")
retriever = RAGRetriever(
    embedding_model=config['embedding']['model_name'],
    db_path=str(project_root / config['data']['vector_db_dir']),
    collection_name=config['vector_db']['collection_name'],
    dense_weight=config['retrieval']['dense_weight'],
    bm25_weight=config['retrieval']['bm25_weight']
)

# Initialize ranker
print("Initializing result ranker...")
ranker = ResultRanker(
    use_cross_encoder=config['retrieval']['use_cross_encoder'],
    model_name=config['retrieval'].get('cross_encoder_model')
)

print(f"\n✅ Retrieval components initialized")
print(f"   Dense weight: {config['retrieval']['dense_weight']}")
print(f"   BM25 weight: {config['retrieval']['bm25_weight']}")
print(f"   Cross-encoder: {config['retrieval']['use_cross_encoder']}")

Phase 5: Retrieval Setup
Initializing hybrid retriever...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8548.46it/s]


Initializing result ranker...

✅ Retrieval components initialized
   Dense weight: 0.7
   BM25 weight: 0.3
   Cross-encoder: False


## Test Retrieval Pipeline

In [12]:
# Test retrieval
test_query = "I need a standing desk for my home office under $300"

print(f"Test Query: '{test_query}'\n")

# Retrieve
retrieved = retriever.retrieve_hybrid(test_query, top_k=config['retrieval']['top_k'])

print(f"Retrieved {len(retrieved)} results from vector database")

# Rank
ranked = ranker.post_process(
    test_query,
    retrieved,
    top_k=5,
    min_confidence=0.0
)

print(f"After ranking and filtering: {len(ranked)} results\n")

# Display results
for i, result in enumerate(ranked, 1):
    print(f"{i}. {result['metadata'].get('product_name', 'Unknown')}")
    print(f"   Category: {result['metadata'].get('category', 'N/A')}")
    print(f"   Price: ${result['metadata'].get('price', 'N/A')}")
    print(f"   Score: {result['combined_score']:.3f}")
    print()

Test Query: 'I need a standing desk for my home office under $300'

Retrieved 20 results from vector database
After ranking and filtering: 5 results

1. IKEA PS 1995
   Category: Home Decor
   Price: $3490.0
   Score: 0.700

2. STOCKHOLM 2025
   Category: Home Decor
   Price: $12990.0
   Score: 0.350

3. EDSBRUK
   Category: Home Decor
   Price: $599.0
   Score: 0.233

4. KNOPPÄNG
   Category: Home Decor
   Price: $3490.0
   Score: 0.175

5. KNOPPÄNG
   Category: Home Decor
   Price: $1990.0
   Score: 0.140



---
# Phase 6: RAG Pipeline

Initialize LangGraph pipeline with LLM integration.

In [13]:
from rag import RAGGraph, AnswerGenerator

print("Phase 6: RAG Pipeline")
print("="*50)

# Initialize generator
print(f"Initializing LLM: {config['llm']['model']}...")

generator = AnswerGenerator(
    llm_provider=config['llm']['provider'],
    model=config['llm']['model'],
    temperature=config['llm']['temperature'],
    max_tokens=config['llm']['max_tokens']
)

print(f"✅ LLM initialized")

# Initialize RAG graph
print(f"\nInitializing LangGraph RAG pipeline...")

rag_graph = RAGGraph(
    retriever=retriever,
    ranker=ranker,
    generator=generator,
    config_path=str(project_root / 'config' / 'prompts.yaml')
)

print(f"✅ RAG pipeline initialized and ready")

Phase 6: RAG Pipeline
Initializing LLM: claude-sonnet-5...
✅ LLM initialized

Initializing LangGraph RAG pipeline...
✅ RAG pipeline initialized and ready


---
# Phase 7: Test Queries

Test the RAG pipeline with sample queries.

In [2]:
import time

print("Phase 7: Query Testing")
print("="*50)

# Test constraint extraction directly
from rag.constraints import extract_price_constraint, extract_and_filter_constraints



# Test queries
test_queries = [
    "I need a comfortable couch under $500",
    "What standing desk options do you have?",
    "I need wall hangings for for small apartments under $200",
]

results = []

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    start = time.time()
    output = rag_graph.invoke(
        query=query,
        top_k=config['retrieval']['top_k'],
        top_k_final=5
    )
    latency = time.time() - start
    
    print(f"\nAnswer:\n{output['answer']}")
    print(f"\nSources ({len(output['sources'])}):")
    for i, source in enumerate(output['sources'], 1):
        print(f"  {i}. {source['product_name']} - ${source['price']}")
    
    print(f"\nLatency: {latency:.2f}s")
    
    results.append({
        'query': query,
        'answer': output['answer'],
        'sources': output['sources'],
        'latency': latency
    })

Phase 7: Query Testing

Query: I need a comfortable couch under $500


No relevant documents found (confidence < 0.1)



[CONSTRAINT FILTER] 20 docs -> 4 docs
[CONSTRAINT FILTER] Applied: {'price': 'under $500', 'min': None, 'max': 500.0}

Answer:
I don't have that information in the catalog.

Sources (4):
  1. RÖDALM - $249.0
  2. KNOPPÄNG - $199.0
  3. KNOPPÄNG - $199.0
  4. RÖDALM - $249.0

Latency: 0.95s

Query: What standing desk options do you have?

[CONSTRAINT FILTER] 20 docs -> 20 docs

Answer:
I don't have that information in the catalog. The provided products are limited to a clock (IKEA PS 1995 - $3490.0) and picture frames (EDSBRUK and KNOPPÄNG in various sizes/colors), with no standing desks listed.

Sources (5):
  1. IKEA PS 1995 - $3490.0
  2. EDSBRUK - $599.0
  3. KNOPPÄNG - $299.0
  4. KNOPPÄNG - $299.0
  5. KNOPPÄNG - $199.0

Latency: 4.04s

Query: I need wall hangings for for small apartments under $200

[CONSTRAINT FILTER] 20 docs -> 3 docs
[CONSTRAINT FILTER] Applied: {'price': 'under $200', 'min': None, 'max': 200.0}

Answer:
Based on the catalog, here's a wall hanging option that

---
# Phase 8: Evaluation (Optional)

Measure system quality metrics.

In [3]:
# Phase 8: Evaluation (LLM-based)
from evaluation import LLMEvaluator
import time

print("Phase 8: Evaluation with LLM")
print("="*50)

# Initialize evaluator
evaluator = LLMEvaluator(model="claude-opus-5")

# Test queries
test_queries = [
    "I need a comfortable couch under $500",
    "What standing desk options do you have?",
    "I need wall hangings for for small apartments under $200",
]

print(f"Evaluating {len(test_queries)} queries...\n")

for query in test_queries:
    print(f"Query: {query}")
    
    start = time.time()
    output = rag_graph.invoke(
        query=query,
        top_k=config['retrieval']['top_k'],
        top_k_final=5
    )
    latency = time.time() - start
    
    # Build context
    context = "\n".join([
        f"{s['product_name']} - ${s['price']}"
        for s in output['sources']
    ])
    
    # Evaluate
    result = evaluator.evaluate(query, output['answer'], context)
    
    print(f"  Faithfulness: {result['faithfulness']:.1%}")
    print(f"  Relevance:    {result['relevance']:.1%}")
    print(f"  Latency:      {latency:.2f}s\n")

# Generate report
print("\n" + "="*50)
print("EVALUATION REPORT")
print("="*50)

if evaluator.results:
    avg_faithfulness = sum(r['faithfulness'] for r in evaluator.results) / len(evaluator.results)
    avg_relevance = sum(r['relevance'] for r in evaluator.results) / len(evaluator.results)
    
    print(f"\nAverage Faithfulness: {avg_faithfulness:.1%}")
    print(f"Average Relevance:    {avg_relevance:.1%}")
    print(f"Average Overall:      {(avg_faithfulness + avg_relevance) / 2:.1%}")

# Save results
evaluator.save_results(str(project_root / 'notebooks' / 'evaluation_results_llm.json'))
print(f"\n✅ Saved evaluation results")



Phase 8: Evaluation with LLM
Evaluating 3 queries...

Query: I need a comfortable couch under $500


No relevant documents found (confidence < 0.1)



[CONSTRAINT FILTER] 20 docs -> 4 docs
[CONSTRAINT FILTER] Applied: {'price': 'under $500', 'min': None, 'max': 500.0}
  Faithfulness: 80.0%
  Relevance:    20.0%
  Latency:      0.37s

Query: What standing desk options do you have?

[CONSTRAINT FILTER] 20 docs -> 20 docs
  Faithfulness: 70.0%
  Relevance:    85.0%
  Latency:      2.15s

Query: I need wall hangings for for small apartments under $200

[CONSTRAINT FILTER] 20 docs -> 3 docs
[CONSTRAINT FILTER] Applied: {'price': 'under $200', 'min': None, 'max': 200.0}
  Faithfulness: 30.0%
  Relevance:    65.0%
  Latency:      3.11s


EVALUATION REPORT

Average Faithfulness: 60.0%
Average Relevance:    56.7%
Average Overall:      58.3%

✅ Saved evaluation results


## Save Results

In [5]:
# Save evaluation results
results_output = project_root / 'notebooks' / 'evaluation_results.json'

# Phase 8: Evaluation (LLM-based)
from evaluation import LLMEvaluator
import time

print("Phase 8: Evaluation with LLM")
print("="*50)

# Initialize evaluator
evaluator = LLMEvaluator(model="claude-opus-5")

# Test queries
test_queries = [
    "I need a comfortable couch under $500",
    "What standing desk options do you have?",
    "Show me bedroom storage solutions",
]

print(f"Evaluating {len(test_queries)} queries...\n")

for query in test_queries:
    print(f"Query: {query}")
    
    start = time.time()
    output = rag_graph.invoke(
        query=query,
        top_k=config['retrieval']['top_k'],
        top_k_final=5
    )
    latency = time.time() - start
    
    # Build context
    context = "\n".join([
        f"{s['product_name']} - ${s['price']}"
        for s in output['sources']
    ])
    
    # Evaluate (skip if no answer)
    if output['answer'] and output['sources']:
        result = evaluator.evaluate(query, output['answer'], context)
        
        print(f"  Faithfulness: {result['faithfulness']:.1%}")
        print(f"  Relevance:    {result['relevance']:.1%}")
    else:
        print(f"  (Skipped - no relevant results)")
    
    print(f"  Latency:      {latency:.2f}s\n")

# Generate report
print("\n" + "="*50)
print("EVALUATION REPORT")
print("="*50)

if evaluator.results:
    avg_faithfulness = sum(r['faithfulness'] for r in evaluator.results) / len(evaluator.results)
    avg_relevance = sum(r['relevance'] for r in evaluator.results) / len(evaluator.results)
    
    print(f"\nQueries evaluated: {len(evaluator.results)}")
    print(f"Average Faithfulness: {avg_faithfulness:.1%}")
    print(f"Average Relevance:    {avg_relevance:.1%}")
    print(f"Average Overall:      {(avg_faithfulness + avg_relevance) / 2:.1%}")
else:
    print("\nNo results to evaluate")

# Save results
evaluator.save_results(str(project_root / 'notebooks' / 'evaluation_results_llm.json'))
print(f"\n✅ Saved evaluation results")
(str(results_output))

print(f"✅ Saved evaluation results to {results_output}")
print(f"\nPipeline execution complete!")

Phase 8: Evaluation with LLM
Evaluating 3 queries...

Query: I need a comfortable couch under $500


No relevant documents found (confidence < 0.1)



[CONSTRAINT FILTER] 20 docs -> 4 docs
[CONSTRAINT FILTER] Applied: {'price': 'under $500', 'min': None, 'max': 500.0}
  Faithfulness: 60.0%
  Relevance:    20.0%
  Latency:      0.45s

Query: What standing desk options do you have?

[CONSTRAINT FILTER] 20 docs -> 20 docs
  Faithfulness: 80.0%
  Relevance:    80.0%
  Latency:      1.91s

Query: Show me bedroom storage solutions

[CONSTRAINT FILTER] 20 docs -> 20 docs
  Faithfulness: 60.0%
  Relevance:    20.0%
  Latency:      1.53s


EVALUATION REPORT

Queries evaluated: 3
Average Faithfulness: 66.7%
Average Relevance:    40.0%
Average Overall:      53.3%

✅ Saved evaluation results
✅ Saved evaluation results to /Users/ramechitturi/coding-projects/gen-academy-agentic-ai/week2-product-search-RAG-ecom-catalog/notebooks/evaluation_results.json

Pipeline execution complete!


---
# Summary

✅ All phases executed successfully!

## What was accomplished:

1. **Ingestion**: Downloaded IKEA dataset (~1000 products)
2. **Cleaning**: Processed and cleaned product data
3. **Chunking**: Split into ~5000-10000 semantic chunks
4. **Embedding**: Generated vector embeddings and populated ChromaDB
5. **Retrieval**: Set up hybrid search (dense + BM25)
6. **RAG**: Integrated LangGraph + Claude LLM
7. **Testing**: Validated with sample queries
8. **Evaluation**: Measured quality metrics

## Next Steps:

- Launch the Streamlit chatbot: `python main.py chatbot`
- Create evaluation dataset with 50+ test queries
- Experiment with different chunking/embedding strategies
- Optimize prompts for better answer quality
- Enable cross-encoder for improved ranking
